# 연습 문제 1-12 풀이: 손실 비례 학습률, 학습률 스케줄링

1-3절 연습 문제 1-12의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 방법으로도 구현할 수 있다.

- 본문 예제 코드는 `code_examples/ch01/` 아래 예제 노트북을 참고한다.

In [1]:
# torch 라이브러리 임포트와 시드 고정
import torch

SEED = 8
torch.manual_seed(SEED)
print(f'시드 고정: {SEED}')

시드 고정: 8


## 사전 준비

본문 1-3절의 데이터 생성, 모델, 손실 함수, 학습 루프([코드 1-28] ~ [코드 1-32])를 가져온다.
에포크마다 로그를 출력하면 실험 조합마다 수백 줄이 나오므로, 마지막 손실만 돌려주는 `train_quiet()`를 따로 구현해 사용한다.

In [2]:
SAMPLE_SIZE = 50    # 생성할 샘플의 수
LR = 0.0001         # 학습률
EPOCHS = 15         # 전체 학습 에포크
INIT = [0.96, -0.56]  # 파라미터 초깃값: 본문 [코드 1-33]과 같은 값

def generate_true_data(sample_size=SAMPLE_SIZE, true_a=4.9, true_b=0.0):
    X = torch.rand((sample_size, 1)) * 10                    # 관측시간
    Y_ideal = true_a * X ** 2 + true_b                       # 이상적인 낙하거리
    noise = torch.randn((sample_size, 1)) * Y_ideal * 0.1    # 관측 오차
    return X, Y_ideal + noise

X, Y_true = generate_true_data()
TRAIN_SIZE = int(SAMPLE_SIZE * 0.8)
X_train, Y_train = X[:TRAIN_SIZE], Y_true[:TRAIN_SIZE]
X_test, Y_test = X[TRAIN_SIZE:], Y_true[TRAIN_SIZE:]
print(f'훈련 데이터 {X_train.shape[0]}개, 평가 데이터 {X_test.shape[0]}개')

훈련 데이터 40개, 평가 데이터 10개


In [3]:
def model(parameters, X):
    return parameters[0] * X ** 2 + parameters[1]

def criterion(Y_pred, Y_true):
    return torch.mean((Y_pred - Y_true) ** 2)

def train_quiet(parameters, X, Y_true, epochs=EPOCHS, learning_rate=LR, criterion=criterion):
    """[코드 1-32]와 같은 학습 루프. 로그 대신 마지막 손실만 반환한다."""
    for _ in range(epochs):
        parameters.grad = None
        train_loss = criterion(model(parameters, X), Y_true)
        train_loss.backward()
        with torch.no_grad():
            parameters -= learning_rate * parameters.grad
    return train_loss.item()

## 연습 문제 1-12 [도전 문제]

> 학습 곡선을 보면 학습 초반에는 손실이 크고, 학습이 진행되면서 점차 줄어든다.
> '손실이 클 때는 강하게, 손실이 작을 때는 약하게 학습한다'는 발상은 딥러닝에만 국한되지 않는 일반적인 원리다.
> 1-3절의 예제에 이 아이디어를 적용해 보자.

**손실에 따라 학습률 조절하기**
- 학습률 = 기준 학습률 * (현재 손실 / 첫 손실)^p
- p = 1이면 손실에 비례하고, p = 0.5이면 손실의 제곱근에 비례
- 손실이 줄어들수록 학습률도 함께 줄어든다

In [4]:
def train_scheduled(parameters, X, Y_true, base_lr, power, epochs=EPOCHS):
    first_loss = None
    for _ in range(epochs):
        parameters.grad = None
        train_loss = criterion(model(parameters, X), Y_true)
        train_loss.backward()
        if first_loss is None:
            first_loss = train_loss.item()
        scale = (train_loss.item() / first_loss) ** power    # 첫 손실 대비 비율
        with torch.no_grad():
            parameters -= base_lr * scale * parameters.grad  # 손실에 따라 조절
    return train_loss.item()

def evaluate(parameters):
    with torch.no_grad():
        return criterion(model(parameters, X_test), Y_test).item()

**고정 학습률과 비교하기**
- 본문의 고정 학습률(0.0001), 손실에 비례(p = 1), 손실의 제곱근에 비례(p = 0.5)
- 훈련 손실과 평가 손실을 함께 비교
- 평가 손실은 학습에 쓰지 않은 데이터로 계산

In [5]:
def fixed(learning_rate):
    return lambda p, e: train_quiet(p, X_train, Y_train, e, learning_rate)

def scheduled(base_lr, power):
    return lambda p, e: train_scheduled(p, X_train, Y_train, base_lr, power, e)

METHODS = [
    ('고정 학습률 0.0001', fixed(0.0001)),
    ('고정 학습률 0.00005', fixed(0.00005)),
    ('손실에 비례, 기준 0.0001', scheduled(0.0001, 1)),
    ('제곱근에 비례, 기준 0.0002', scheduled(0.0002, 0.5)),
]

def compare(epochs):
    print(f'{"훈련 손실":>9} {"평가 손실":>9} | 방식 ({epochs}에포크)')
    for name, run in METHODS:
        parameters = torch.tensor(INIT, requires_grad=True)
        train_loss = run(parameters, epochs)
        print(f'{train_loss:13.2f} {evaluate(parameters):13.2f} | {name}')

In [6]:
compare(EPOCHS)

    훈련 손실     평가 손실 | 방식 (15에포크)
       380.40        805.19 | 고정 학습률 0.0001
       557.34        569.61 | 고정 학습률 0.00005
      2682.90       2556.03 | 손실에 비례, 기준 0.0001
       477.10        585.07 | 제곱근에 비례, 기준 0.0002


**결과 분석**
- 손실에 비례시키면 학습이 크게 느려진다: 15에포크 뒤에도 훈련 손실이 2,683
- 제곱근에 비례시키면 평가 손실이 805에서 585로 내려간다
- 그런데 고정 학습률 0.00005도 평가 손실이 570으로 낮다

**왜 손실에 그대로 비례시키면 느려질까? 기울기를 보자**
- 평균제곱오차의 기울기는 오차에 비례하므로, 손실이 줄면 기울기도 함께 줄어든다
- 고정 학습률로 학습해도 걸음 폭이 이미 저절로 줄어드는 중

In [7]:
parameters = torch.tensor(INIT, requires_grad=True)
for epoch in range(1, EPOCHS + 1):
    parameters.grad = None
    train_loss = criterion(model(parameters, X_train), Y_train)
    train_loss.backward()
    if epoch in (1, 8, EPOCHS):
        loss, grad_a = train_loss.item(), parameters.grad[0].item()
        print(f'{epoch:2d}번째 에포크: 손실 {loss:9.2f}, a의 기울기 {grad_a:10.2f}')
    with torch.no_grad():
        parameters -= LR * parameters.grad

 1번째 에포크: 손실  29287.49, a의 기울기  -13866.57
 8번째 에포크: 손실    480.44, a의 기울기    -817.13
15번째 에포크: 손실    380.40, a의 기울기     -48.14


**정리: 손실이 이미 걸음 폭에 한 번 반영되어 있다**
- 학습률까지 손실에 비례시키면 같은 효과가 두 번 곱해진다
- 제곱근을 씌운 것은 이 이중 반영을 줄이기 위해서

**평가 손실이 낮아진 것은 스케줄링 덕분일까? 100에포크로 늘려 보자**
- 15에포크의 평가 손실은 학습이 덜 진행된 상태에서 잰 값
- 같은 방식들을 100에포크로 학습해 비교

In [8]:
compare(100)

    훈련 손실     평가 손실 | 방식 (100에포크)
       380.05        821.24 | 고정 학습률 0.0001
       380.05        821.25 | 고정 학습률 0.00005
       641.47        591.86 | 손실에 비례, 기준 0.0001
       380.05        820.82 | 제곱근에 비례, 기준 0.0002


**확인**
- 100에포크에서는 고정 학습률과 제곱근 방식이 훈련 손실 380, 평가 손실 821로 같은 곳에 도착
- 손실에 비례시킨 방식은 100에포크에서도 훈련 손실이 641로 덜 학습된 상태이고, 평가 손실이 낮은 것도 같은 이유
- 15에포크에서 평가 손실이 낮았던 것은 학습이 덜 된 덕분이고, 이 예제에서는 스케줄링의 이점이 확인되지 않는다
- 학습을 더 하면 평가 손실이 오히려 오르는 것은 학습률이 아닌 과적합 문제

**직접 해 보기: 손실에 비례시키고 기준 학습률을 0.0008로 키우면?**
- 기준 학습률이 크면 더 빨리 학습될까?
- 먼저 손실이 어떻게 변할지 예측해 보자

In [9]:
parameters = torch.tensor(INIT, requires_grad=True)
first_loss = criterion(model(parameters, X_train), Y_train).item()
print(f'{"에포크":>5} {"훈련 손실":>9} | 학습률')
for epoch in range(1, 6):
    parameters.grad = None
    train_loss = criterion(model(parameters, X_train), Y_train)
    train_loss.backward()
    learning_rate = 0.0008 * train_loss.item() / first_loss
    print(f'{epoch:8d} {train_loss.item():13.3e} | {learning_rate:.4g}')
    with torch.no_grad():
        parameters -= learning_rate * parameters.grad

  에포크     훈련 손실 | 학습률
       1     2.929e+04 | 0.0008
       2     8.018e+04 | 0.00219
       3     3.153e+06 | 0.08614
       4     2.571e+11 | 7024
       5     1.404e+26 | 3.835e+18


**확인**
- 2에포크에서 손실이 오히려 커지고, 손실에 비례한 학습률도 함께 커진다
- 커진 학습률이 다음 걸음을 더 키워 손실이 걷잡을 수 없이 커진다(계속하면 `inf`, `nan`)
- 손실에 비례시키는 방식은 기준 학습률을 신중하게 잡아야 한다